# Decision Tree Encoding Benchmark

## Objective

Evaluate categorical encoding strategies for Decision Tree Regression.

## Encoders Evaluated

- OneHotEncoder
- OrdinalEncoder

## Evaluation Criteria

- Feature dimensionality
- Memory consumption
- Training time
- Model suitability

## Conclusion

OrdinalEncoder was selected because Decision Trees are insensitive to monotonic transformations of categorical labels and do not rely on distance calculations. It provides a significantly lower-dimensional feature space while maintaining efficient model performance.

---

## Output

The selected encoding strategy was adopted in the final Decision Tree Regression pipeline.

In [1]:
import sqlite3
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder
)

from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

In [2]:
#######################################
# Load dataset
#######################################

print("Loading dataset...")

# Connect to database
conn = sqlite3.connect("pakwheels_dataset.db")

df = pd.read_sql_query(
    "SELECT * FROM market_analysis_cars",
    conn
)

conn.close()

# Keep only the columns used by the Decision Tree model
required_columns = [
    "Company",
    "Model",
    "City",
    "Year",
    "Mileage_KM",
    "Engine_CC",
    "Fuel_Type",
    "Transmission",
    "Price_PKR"
]

df = df[required_columns].dropna()

#######################################
# Features and target
#######################################

X = df.drop("Price_PKR", axis=1)
y = df["Price_PKR"]

#######################################
# Column definitions
#######################################

categorical_columns = [
    "Company",
    "Model",
    "City",
    "Fuel_Type",
    "Transmission"
]

numeric_columns = [
    "Year",
    "Mileage_KM",
    "Engine_CC"
]

#######################################
# Benchmark settings
#######################################

NUM_RUNS = 30

print(f"Dataset Loaded Successfully")
print(f"Records: {len(df)}")
print(f"Benchmark Runs: {NUM_RUNS}")

Loading dataset...
Dataset Loaded Successfully
Records: 2526
Benchmark Runs: 30


In [3]:
#######################################
# Benchmark Decision Tree Encoders
#######################################

def run_tree_encoder_benchmark(encoder_name, encoder):

    preprocessing_times = []
    fit_times = []
    prediction_times = []
    total_times = []

    feature_dimensions = []
    memory_usage = []

    mae_scores = []
    r2_scores = []

    for run in range(NUM_RUNS + 1):

        ###################################
        # Build preprocessing pipeline
        ###################################

        preprocessor = ColumnTransformer(
            transformers=[
                ("cat", encoder, categorical_columns),
                ("num", "passthrough", numeric_columns)
            ],
            remainder="drop"
        )

        ###################################
        # Build Decision Tree
        ###################################

        regression_tree = DecisionTreeRegressor(
            max_depth=8,
            min_samples_leaf=10,
            random_state=42
        )

        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("regressor", regression_tree)
            ]
        )

        ###################################
        # Train/Test Split
        ###################################

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )

        ###################################
        # Warm-up iteration
        ###################################

        if run == 0:

            model.fit(X_train, y_train)
            model.predict(X_test)

            continue

        ###################################
        # Total Timer
        ###################################

        total_start = time.perf_counter()

        ###################################
        # Preprocessing
        ###################################

        preprocessing_start = time.perf_counter()

        preprocessor.fit(X_train)

        X_train_processed = preprocessor.transform(X_train)
        X_test_processed = preprocessor.transform(X_test)

        preprocessing_end = time.perf_counter()

        ###################################
        # Feature Information
        ###################################

        feature_dimensions.append(
            X_train_processed.shape[1]
        )

        if hasattr(X_train_processed, "nbytes"):

            memory_usage.append(
                X_train_processed.nbytes / (1024 ** 2)
            )

        else:

            memory_usage.append(
                X_train_processed.data.nbytes / (1024 ** 2)
            )

        ###################################
        # Fit Timer
        ###################################

        fit_start = time.perf_counter()

        model.fit(X_train, y_train)

        fit_end = time.perf_counter()

        ###################################
        # Prediction Timer
        ###################################

        prediction_start = time.perf_counter()

        predictions = model.predict(X_test)

        prediction_end = time.perf_counter()

        ###################################
        # Accuracy
        ###################################

        mae_scores.append(
            mean_absolute_error(
                y_test,
                predictions
            )
        )

        r2_scores.append(
            r2_score(
                y_test,
                predictions
            )
        )

        ###################################
        # Total Timer
        ###################################

        total_end = time.perf_counter()

        preprocessing_times.append(
            preprocessing_end - preprocessing_start
        )

        fit_times.append(
            fit_end - fit_start
        )

        prediction_times.append(
            prediction_end - prediction_start
        )

        total_times.append(
            total_end - total_start
        )

    return {

        "Encoder": encoder_name,

        "Feature Dimensions": feature_dimensions,
        "Memory (MB)": memory_usage,

        "Preprocessing Time": preprocessing_times,
        "Fit Time": fit_times,
        "Prediction Time": prediction_times,
        "Total Time": total_times,

        "MAE": mae_scores,
        "R2": r2_scores
    }

In [4]:
#######################################
# Benchmark OneHotEncoder
#######################################

onehot_results = run_tree_encoder_benchmark(
    encoder_name="OneHotEncoder",
    encoder=OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
)

print("OneHotEncoder Benchmark Completed")

OneHotEncoder Benchmark Completed


In [5]:
#######################################
# Benchmark OrdinalEncoder
#######################################

ordinal_results = run_tree_encoder_benchmark(
    encoder_name="OrdinalEncoder",
    encoder=OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
)

print("OrdinalEncoder Benchmark Completed")

OrdinalEncoder Benchmark Completed


In [6]:
#######################################
# Display Raw Benchmark Results
#######################################

print("=" * 60)
print("OneHotEncoder Results")
print("=" * 60)

for key, value in onehot_results.items():
    if isinstance(value, list):
        print(f"\n{key}:")
        print(value)
    else:
        print(f"{key}: {value}")

print("\n" + "=" * 60)
print("OrdinalEncoder Results")
print("=" * 60)

for key, value in ordinal_results.items():
    if isinstance(value, list):
        print(f"\n{key}:")
        print(value)
    else:
        print(f"{key}: {value}")

OneHotEncoder Results
Encoder: OneHotEncoder

Feature Dimensions:
[199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199, 199]

Memory (MB):
[3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875, 3.066864013671875]

Preprocessing Time:
[0.021713399997679517, 0.02253259999997681, 0.02152360000036424, 0.019887099995685276, 0.01871209999808343, 0.02245950000360608, 0.02067199999873992, 0.019971699999587145, 0.018

In [7]:
#######################################
# Summarize Encoder Benchmark Results
#######################################

def summarize_results(results):
    return {
        "Encoder": results["Encoder"],

        "Feature Dimensions": int(np.mean(results["Feature Dimensions"])),
        "Memory (MB)": round(np.mean(results["Memory (MB)"]), 6),

        "Preprocessing Mean (s)": round(np.mean(results["Preprocessing Time"]), 6),
        "Preprocessing Std (s)": round(np.std(results["Preprocessing Time"]), 6),
        "Preprocessing Min (s)": round(np.min(results["Preprocessing Time"]), 6),
        "Preprocessing Max (s)": round(np.max(results["Preprocessing Time"]), 6),

        "Fit Mean (s)": round(np.mean(results["Fit Time"]), 6),
        "Fit Std (s)": round(np.std(results["Fit Time"]), 6),
        "Fit Min (s)": round(np.min(results["Fit Time"]), 6),
        "Fit Max (s)": round(np.max(results["Fit Time"]), 6),

        "Prediction Mean (s)": round(np.mean(results["Prediction Time"]), 6),
        "Prediction Std (s)": round(np.std(results["Prediction Time"]), 6),
        "Prediction Min (s)": round(np.min(results["Prediction Time"]), 6),
        "Prediction Max (s)": round(np.max(results["Prediction Time"]), 6),

        "Total Mean (s)": round(np.mean(results["Total Time"]), 6),
        "Total Std (s)": round(np.std(results["Total Time"]), 6),
        "Total Min (s)": round(np.min(results["Total Time"]), 6),
        "Total Max (s)": round(np.max(results["Total Time"]), 6)
    }


summary_df = pd.DataFrame([
    summarize_results(onehot_results),
    summarize_results(ordinal_results)
])

print("\n========== Decision Tree Encoder Benchmark Summary ==========\n")

display(summary_df)


========== Decision Tree Encoder Benchmark Summary ==========



,Encoder,Feature Dimensions,Memory (MB),Preprocessing Mean (s),Preprocessing Std (s),Preprocessing Min (s),Preprocessing Max (s),Fit Mean (s),Fit Std (s),Fit Min (s),Fit Max (s),Prediction Mean (s),Prediction Std (s),Prediction Min (s),Prediction Max (s),Total Mean (s),Total Std (s),Total Min (s),Total Max (s)
0,OneHotEncoder,199,3.066864,0.023398,0.003134,0.018712,0.028862,0.023623,0.003544,0.019177,0.031966,0.005454,0.000950,0.00409,0.008693,0.053532,0.007218,0.043342,0.069032
1,OrdinalEncoder,8,0.123291,0.021244,0.005149,0.016748,0.038404,0.013584,0.002256,0.010598,0.022270,0.005362,0.000833,0.00416,0.007340,0.041321,0.007502,0.034170,0.069698
